# 🔌 European Electricity Interconnection: Grid Risk Analysis

**Analyst Perspective:** Infrastructure Risk Manager / Energy Security Analyst

This notebook analyzes Europe's cross-border electricity interconnection from a risk and resilience perspective. It covers interconnector capacity, import dependency, renewable penetration, and projected infrastructure needs through 2040.

**Key Risk Dimensions:**
- Interconnector capacity adequacy and bottlenecks
- Import dependency risk by country
- Renewable energy penetration vs grid readiness
- Scenario analysis: Reference vs Projects vs Needs
- Peak demand vs interconnection adequacy
- Cross-border flow vulnerability

**Data Source:** European Electricity Interconnection Data (Ember Energy)

In [ ]:
# ============================================================
# 1. IMPORTS & SETUP
# ============================================================
import warnings


warnings.filterwarnings("ignore")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


print("All imports successful.")

In [ ]:
# ============================================================
# 2. KAGGLE DATA LOADING
# ============================================================
import os


if os.path.exists("/kaggle/input"):
    BASE = "/kaggle/input/european-electricity-interconnection"
else:
    BASE = "../../EuropeanElectricty"

print(f"Data base path: {BASE}")

In [ ]:
# ============================================================
# 3. LOAD ALL INTERCONNECTION DATASETS
# ============================================================
def load_csv(name):
    path = f"{BASE}/csv/{name}.csv"
    df = pd.read_csv(path)
    print(f"  {name}: {len(df):,} rows x {len(df.columns)} cols")
    return df


print("Loading interconnection data...")
peak_demand = load_csv("peak_demand")
interconnectors = load_csv("interconnectors")
import_potential = load_csv("import_potential")
country_indicators = load_csv("country_indicators")
flow_indicators = load_csv("flow_indicators")

print(
    f"\nTotal: {sum(len(d) for d in [peak_demand, interconnectors, import_potential, country_indicators, flow_indicators]):,} rows"
)

---
## 📡 Risk Analysis 1: Interconnector Capacity & Bottlenecks

The European grid relies on cross-border interconnectors. We identify critical corridors, capacity gaps, and the gap between current (Reference), planned (Projects), and required (Needs) capacity.

In [ ]:
# ============================================================
# 4. INTERCONNECTOR CAPACITY BY SCENARIO
# ============================================================
fig = px.box(
    interconnectors,
    x="Scenario",
    y="NTC_F",
    color="Scenario",
    points="all",
    title="Interconnector Capacity (NTC_F) Distribution by Scenario — All Borders",
    labels={"NTC_F": "Forward NTC (MW)", "Scenario": ""},
    template="plotly_white",
    height=500,
)
fig.update_layout(showlegend=False)
fig.show()

scenario_stats = interconnectors.groupby("Scenario")["NTC_F"].describe().round(0)
print("\nScenario Capacity Statistics (MW):")
scenario_stats

In [ ]:
# ============================================================
# 5. TOP INTERCONNECTORS BY CAPACITY
# ============================================================
# Reference scenario (current state)
ref = interconnectors[interconnectors["Scenario"] == "Reference"].copy()
ref["Total_NTC"] = ref["NTC_F"] + ref["NTC_B"]
top_ref = ref.sort_values("NTC_F", ascending=False).head(20)

fig = px.bar(
    top_ref,
    x="NTC_F",
    y="Border",
    orientation="h",
    color="NTC_F",
    color_continuous_scale="Viridis",
    title="Top 20 Interconnectors — Reference Capacity (MW)",
    labels={"NTC_F": "Forward NTC (MW)", "Border": ""},
    template="plotly_white",
    height=600,
    hover_data=["From", "To", "NTC_B"],
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

In [ ]:
# ============================================================
# 6. CAPACITY GAP ANALYSIS: Projects vs Needs
# ============================================================
proj = interconnectors[interconnectors["Scenario"] == "Projects"][
    ["Border", "From", "To", "NTC_F", "NTC_B"]
].copy()
needs = interconnectors[interconnectors["Scenario"] == "Needs"][
    ["Border", "From", "To", "NTC_F", "NTC_B"]
].copy()
ref_small = interconnectors[interconnectors["Scenario"] == "Reference"][["Border", "NTC_F"]].copy()

gap = proj.merge(needs, on="Border", suffixes=("_Proj", "_Needs"))
gap = gap.merge(ref_small, on="Border")
gap["Gap_NTC_F"] = gap["NTC_F_Needs"] - gap["NTC_F_Proj"]
gap["Increase_From_Ref"] = gap["NTC_F_Proj"] - gap["NTC_F"]
gap = gap.sort_values("Gap_NTC_F", ascending=False)

top_gaps = gap.head(15)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=top_gaps["Border"],
        x=top_gaps["NTC_F"],
        name="Reference",
        orientation="h",
        marker_color="#1f77b4",
    )
)
fig.add_trace(
    go.Bar(
        y=top_gaps["Border"],
        x=top_gaps["NTC_F_Proj"] - top_gaps["NTC_F"],
        name="Planned Addition (Projects)",
        orientation="h",
        marker_color="#ff7f0e",
        base=top_gaps["NTC_F"],
    )
)
fig.add_trace(
    go.Bar(
        y=top_gaps["Border"],
        x=top_gaps["Gap_NTC_F"],
        name="Additional Need (Needs - Projects)",
        orientation="h",
        marker_color="#d62728",
        base=top_gaps["NTC_F_Proj"],
    )
)

fig.update_layout(
    title="Interconnector Capacity Gap: Reference → Projects → Needs (Top 15 Borders)",
    xaxis_title="NTC Forward Capacity (MW)",
    barmode="stack",
    height=600,
    template="plotly_white",
    legend=dict(x=0.7, y=0.05),
)
fig.update_yaxes(autorange="reversed")
fig.show()

print("\nTotal identified capacity gap:")
print(f"  Currently planned increase: {gap['Increase_From_Ref'].sum():,.0f} MW")
print(f"  Additional need:             {gap['Gap_NTC_F'].sum():,.0f} MW")
print(
    f"  Total required from today:   {(gap['Increase_From_Ref'] + gap['Gap_NTC_F']).sum():,.0f} MW"
)

---
## ⚠️ Risk Analysis 2: Import Dependency Risk

Import potential measures how much of a country's peak demand could be met through imports. High values indicate critical infrastructure dependency.

In [ ]:
# ============================================================
# 7. IMPORT DEPENDENCY RISK
# ============================================================
fig = px.bar(
    import_potential,
    x="Import Potential (%)",
    y="Country",
    color="Scenario",
    orientation="h",
    title="Import Potential (%) by Country & Scenario — 2030 & 2040 Horizons",
    labels={"Import Potential (%)": "Import Potential (%)", "Country": ""},
    template="plotly_white",
    height=900,
    barmode="group",
    facet_row="Target Year",
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"],
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

In [ ]:
# ============================================================
# 8. CRITICAL DEPENDENCY COUNTRIES
# ============================================================
# Countries with import potential > 100% are critically dependent on imports
critical = import_potential[import_potential["Import Potential (%)"] > 100].copy()
critical = critical.sort_values("Import Potential (%)", ascending=False)

fig = px.bar(
    critical,
    x="Import Potential (%)",
    y="Country",
    color="Target Year",
    orientation="h",
    title="🚨 CRITICAL IMPORT DEPENDENCY: Countries Where Import Potential > 100%",
    labels={"Import Potential (%)": "Import Potential (%)", "Country": ""},
    template="plotly_white",
    height=500,
    barmode="group",
    pattern_shape="Scenario",
    color_discrete_sequence=["#d62728", "#ff7f0e"],
)
fig.add_vline(
    x=100,
    line_dash="dash",
    line_color="black",
    annotation_text="100% Threshold",
    annotation_position="top right",
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

print("\nCountries critically dependent on imports (Potential > 100%):")
for _, row in critical.iterrows():
    print(
        f"  {row['Country']:25s}  {row['Import Potential (%)']:>8.1f}%  ({row['Scenario']:15s}  {int(row['Target Year'])})"
    )

In [ ]:
# ============================================================
# 9. PEAK DEMAND VS IMPORT CAPACITY
# ============================================================
# Merge peak demand with import potential (using Reference scenario, 2030 target)
imp_2030_ref = import_potential[
    (import_potential["Target Year"] == 2030) & (import_potential["Scenario"] == "Reference")
].copy()

# Map full country names to ISO2 codes using a manual mapping
country_to_iso2 = {
    "Albania": "AL",
    "Austria": "AT",
    "Belgium": "BE",
    "Bulgaria": "BG",
    "Croatia": "HR",
    "Cyprus": "CY",
    "Czechia": "CZ",
    "Denmark": "DK",
    "Estonia": "EE",
    "Finland": "FI",
    "France": "FR",
    "Germany": "DE",
    "Greece": "GR",
    "Hungary": "HU",
    "Ireland": "IE",
    "Italy": "IT",
    "Latvia": "LV",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Malta": "MT",
    "Montenegro": "ME",
    "Netherlands": "NL",
    "North Macedonia": "MK",
    "Norway": "NO",
    "Poland": "PL",
    "Portugal": "PT",
    "Romania": "RO",
    "Serbia": "RS",
    "Slovakia": "SK",
    "Slovenia": "SI",
    "Spain": "ES",
    "Sweden": "SE",
    "Switzerland": "CH",
    "United Kingdom": "UK",
}
imp_2030_ref["ISO2"] = imp_2030_ref["Country"].map(country_to_iso2)

peak_2030 = peak_demand[peak_demand["Year"] == 2030].copy()

merged = imp_2030_ref.merge(
    peak_2030, left_on="ISO2", right_on="Country", how="inner", suffixes=("", "_peak")
)
merged["Peak Demand (GW)"] = merged["Peak Demand (MW)"] / 1000
merged = merged.dropna(subset=["Import Potential (%)", "Peak Demand (GW)"])

fig = px.scatter(
    merged,
    x="Peak Demand (GW)",
    y="Import Potential (%)",
    text="Country",
    size="Peak Demand (GW)",
    title="2030 Reference Scenario: Peak Demand vs Import Dependency",
    labels={"Peak Demand (GW)": "Peak Demand (GW)", "Import Potential (%)": "Import Potential (%)"},
    template="plotly_white",
    height=600,
    hover_data=["Import Potential (%)"],
)
fig.add_hline(
    y=100, line_dash="dash", line_color="red", annotation_text="100% Dependency Threshold"
)
fig.update_traces(textposition="top center", marker=dict(line=dict(width=1, color="DarkSlateGrey")))
fig.show()

print("\nLarge economies with high import dependency:")
large_dependent = merged[(merged["Import Potential (%)"] > 50) & (merged["Peak Demand (GW)"] > 20)]
for _, row in large_dependent.sort_values("Import Potential (%)", ascending=False).iterrows():
    print(
        f"  {row['Country']:20s}  Peak: {row['Peak Demand (GW)']:>6.1f} GW  Import Potential: {row['Import Potential (%)']:>6.1f}%"
    )

---
## 🌱 Risk Analysis 3: Renewable Penetration & Grid Readiness

High renewable penetration without adequate interconnection creates curtailment and stability risks.

In [ ]:
# ============================================================
# 10. RENEWABLE PENETRATION (RES-E) ANALYSIS
# ============================================================
# Monthly aggregation, 2024 actual data
ci_monthly = country_indicators[
    (country_indicators["Aggregation"] == "monthly") & (country_indicators["Scenario"] == "2024")
].copy()

res_by_country = (
    ci_monthly.groupby("Country")["RES-E"].agg(["mean", "min", "max", "std"]).reset_index()
)
res_by_country.columns = ["Country", "RES-E Mean", "RES-E Min", "RES-E Max", "RES-E Std"]
res_by_country = res_by_country.sort_values("RES-E Mean", ascending=False)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Average RES-E % (2024)", "RES-E Volatility (Std Dev)"),
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Bar(
        y=res_by_country["Country"],
        x=res_by_country["RES-E Mean"],
        orientation="h",
        marker_color="#2ca02c",
        hovertemplate="%{y}: %{x:.1f}%<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        y=res_by_country["Country"],
        x=res_by_country["RES-E Std"],
        orientation="h",
        marker_color="#ff7f0e",
        hovertemplate="%{y}: %{x:.1f}%<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    height=700,
    title_text="Renewable Energy Share (RES-E) by Country — 2024",
    template="plotly_white",
    showlegend=False,
)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_xaxes(title_text="RES-E (%)", row=1, col=1)
fig.update_xaxes(title_text="Std Dev (%)", row=1, col=2)
fig.show()

In [ ]:
# ============================================================
# 11. RES-E VS NET POSITION (Are RES leaders net importers or exporters?)
# ============================================================
net_by_country = ci_monthly.groupby("Country")[["RES-E", "NET-P"]].mean().reset_index()

fig = px.scatter(
    net_by_country,
    x="RES-E",
    y="NET-P",
    text="Country",
    title="Renewable Penetration vs Net Position — 2024 Monthly Averages",
    labels={
        "RES-E": "RES-E (%) — Renewable Share",
        "NET-P": "Net Position (GW) — Positive = Net Exporter",
    },
    template="plotly_white",
    height=600,
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.update_traces(
    textposition="top center",
    marker=dict(size=10, color="#1f77b4", line=dict(width=1, color="DarkSlateGrey")),
)
fig.show()

print("\nNet Exporters (NET-P > 5 GW):")
exporters = net_by_country[net_by_country["NET-P"] > 5].sort_values("NET-P", ascending=False)
for _, r in exporters.iterrows():
    print(f"  {r['Country']:20s}  RES-E: {r['RES-E']:5.1f}%  NET-P: {r['NET-P']:+.1f} GW")

print("\nNet Importers (NET-P < -5 GW):")
importers = net_by_country[net_by_country["NET-P"] < -5].sort_values("NET-P")
for _, r in importers.iterrows():
    print(f"  {r['Country']:20s}  RES-E: {r['RES-E']:5.1f}%  NET-P: {r['NET-P']:+.1f} GW")

In [ ]:
# ============================================================
# 12. DIURNAL RENEWABLE PATTERN (Hourly RES-E)
# ============================================================
ci_hourly = country_indicators[
    (country_indicators["Aggregation"] == "hourly") & (country_indicators["Scenario"] == "2024")
].copy()
ci_hourly["Time_Period"] = pd.to_numeric(ci_hourly["Time_Period"], errors="coerce")

top_res = res_by_country[res_by_country["RES-E Mean"] > 40]["Country"].tolist()[:6]
hourly_res = ci_hourly[
    ci_hourly["Country"].isin(
        ["Denmark", "Ireland", "Portugal", "United Kingdom", "Germany", "Spain"]
    )
]

fig = px.line(
    hourly_res,
    x="Time_Period",
    y="RES-E",
    color="Country",
    title="Diurnal RES-E Pattern — Selected High-Renewable Countries (2024)",
    labels={"Time_Period": "Hour of Day", "RES-E": "RES-E (%)"},
    template="plotly_white",
    height=450,
    markers=True,
)
fig.update_layout(xaxis=dict(tickmode="linear", tick0=1, dtick=2))
fig.show()

---
## 🌊 Risk Analysis 4: Cross-Border Flow Vulnerability

Analyzing who sends power where reveals systemic risks — single points of failure in the grid.

In [ ]:
# ============================================================
# 13. CROSS-BORDER FLOW ANALYSIS (Monthly, 2024)
# ============================================================
flow_monthly = flow_indicators[
    (flow_indicators["Aggregation"] == "monthly") & (flow_indicators["Year"] == 2024)
].copy()

# Aggregate by direction (from → to)
flow_summary = flow_monthly.groupby(["Country_From", "Country_To"])["Flow"].mean().reset_index()

# Top export flows
top_exports = flow_summary.sort_values("Flow", ascending=False).head(15)

fig = px.bar(
    top_exports,
    x="Flow",
    y="Country_From",
    orientation="h",
    color="Country_To",
    title="Top 15 Cross-Border Export Flows — 2024 Monthly Average (MW)",
    labels={"Flow": "Average Flow (MW)", "Country_From": "Exporting Country"},
    template="plotly_white",
    height=500,
    hover_data=["Country_To"],
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

In [ ]:
# ============================================================
# 14. NET EXPORTER / IMPORTER CLASSIFICATION
# ============================================================
# Compute net position for each country
flow_from = flow_summary.groupby("Country_From")["Flow"].sum().reset_index()
flow_to = flow_summary.groupby("Country_To")["Flow"].sum().reset_index()
flow_from.columns = ["Country", "Exports"]
flow_to.columns = ["Country", "Imports"]

net = flow_from.merge(flow_to, on="Country", how="outer").fillna(0)
net["Imports"] = net["Imports"].abs()
net["Net_Flow"] = net["Exports"] - net["Imports"]
net = net.sort_values("Net_Flow", ascending=False)

fig = px.bar(
    net,
    x="Net_Flow",
    y="Country",
    orientation="h",
    color="Net_Flow",
    color_continuous_scale="RdYlBu",
    title="Net Flow Position by Country — 2024 (Positive = Net Exporter)",
    labels={"Net_Flow": "Net Flow (MW)", "Country": ""},
    template="plotly_white",
    height=700,
)
fig.add_vline(x=0, line_dash="solid", line_color="black", line_width=1)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

print("\nTop 5 Net Exporters:")
for _, r in net.head(5).iterrows():
    print(
        f"  {r['Country']:20s}  Net: {r['Net_Flow']:+.1f} MW  (Export: {r['Exports']:.0f} | Import: {r['Imports']:.0f})"
    )

print("\nTop 5 Net Importers:")
for _, r in net.tail(5).sort_values("Net_Flow").iterrows():
    print(
        f"  {r['Country']:20s}  Net: {r['Net_Flow']:+.1f} MW  (Export: {r['Exports']:.0f} | Import: {r['Imports']:.0f})"
    )

In [ ]:
# ============================================================
# 15. FLOW HEATMAP: Who sends power where
# ============================================================
flow_pivot = flow_summary.pivot(index="Country_From", columns="Country_To", values="Flow").fillna(0)

# Filter to most active corridors
top_corridors = (
    flow_summary.groupby("Country_From")["Flow"].sum().sort_values(ascending=False).head(15).index
)
flow_filtered = flow_pivot.loc[flow_pivot.index.intersection(top_corridors), :]
flow_filtered = flow_filtered.loc[
    :, flow_filtered.sum().sort_values(ascending=False).head(15).index
]

fig = px.imshow(
    flow_filtered,
    labels=dict(x="Importing Country", y="Exporting Country", color="Avg Flow (MW)"),
    title="Cross-Border Flow Matrix — Top 15 Countries (2024 Monthly Avg, MW)",
    color_continuous_scale="RdYlBu_r",
    aspect="auto",
    height=600,
    template="plotly_white",
    text_auto=".0f",
)
fig.update_layout(coloraxis_colorbar_title="MW")
fig.show()

---
## 📈 Risk Analysis 5: Temporal & Scenario Projections

How does the grid risk evolve from 2024 through 2030 to 2040?

In [ ]:
# ============================================================
# 16. PEAK DEMAND GROWTH PROJECTION
# ============================================================
peak_pivot = peak_demand.pivot(index="Country", columns="Year", values="Peak Demand (MW)")
peak_pivot["Growth_24_40"] = (peak_pivot[2040] - peak_pivot[2024]) / peak_pivot[2024] * 100
peak_pivot = peak_pivot.dropna(subset=["Growth_24_40"]).sort_values("Growth_24_40", ascending=False)

top_growth = peak_pivot.head(15).reset_index()

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=top_growth["Country"],
        x=top_growth["Growth_24_40"],
        orientation="h",
        marker_color=top_growth["Growth_24_40"],
        marker_colorscale="RdYlBu_r",
        hovertemplate="%{y}: %{x:.1f}% growth<extra></extra>",
    )
)
fig.update_layout(
    title="Projected Peak Demand Growth 2024→2040 — Top 15 Countries",
    xaxis_title="Growth (%)",
    yaxis_title="",
    template="plotly_white",
    height=500,
)
fig.update_yaxes(autorange="reversed")
fig.show()

print("\nPeak Demand Growth Projections:")
print(f"  Average growth (2024→2040): {peak_pivot['Growth_24_40'].mean():.1f}%")
print(f"  Fastest growing: {peak_pivot.index[0]} ({peak_pivot['Growth_24_40'].iloc[0]:.1f}%)")
print(f"  Fastest declining: {peak_pivot.index[-1]} ({peak_pivot['Growth_24_40'].iloc[-1]:.1f}%)")

In [ ]:
# ============================================================
# 17. RES-E EVOLUTION: 2024 → 2030 → 2040
# ============================================================
res_by_year = (
    country_indicators[
        (country_indicators["Aggregation"] == "monthly")
        & (country_indicators["Scenario"].isin(["2024", "2030", "2040"]))
    ]
    .groupby(["Country", "Scenario"])["RES-E"]
    .mean()
    .reset_index()
)

top_res_countries = (
    res_by_year[res_by_year["Scenario"] == "2024"]
    .sort_values("RES-E", ascending=False)
    .head(8)["Country"]
    .tolist()
)
res_selected = res_by_year[res_by_year["Country"].isin(top_res_countries)]

fig = px.line(
    res_selected,
    x="Scenario",
    y="RES-E",
    color="Country",
    markers=True,
    title="RES-E Trajectory: Top 8 Countries by 2024 Penetration",
    labels={"Scenario": "Year / Scenario", "RES-E": "RES-E (%)"},
    template="plotly_white",
    height=500,
)
fig.show()

print("\nRES-E growth from 2024 baseline to 2040 projection:")
for c in top_res_countries:
    v24 = res_by_year[(res_by_year["Country"] == c) & (res_by_year["Scenario"] == "2024")][
        "RES-E"
    ].values
    v40 = res_by_year[(res_by_year["Country"] == c) & (res_by_year["Scenario"] == "2040")][
        "RES-E"
    ].values
    if len(v24) and len(v40):
        print(f"  {c:20s}: {v24[0]:5.1f}% → {v40[0]:5.1f}%  (Δ {v40[0] - v24[0]:+.1f}pp)")

In [ ]:
# ============================================================
# 18. IMPORT POTENTIAL GAP ANALYSIS: Today vs 2040 Needs
# ============================================================
imp_2024 = import_potential[import_potential["Scenario"] == "2024"][
    ["Country", "Import Potential (%)"]
].copy()
imp_2040_needs = import_potential[
    (import_potential["Target Year"] == 2040) & (import_potential["Scenario"] == "Needs")
][["Country", "Import Potential (%)"]].copy()

imp_gap = imp_2024.merge(imp_2040_needs, on="Country", suffixes=("_2024", "_2040_Needs"))
imp_gap["Change"] = (
    imp_gap["Import Potential (%)_2040_Needs"] - imp_gap["Import Potential (%)_2024"]
)
imp_gap = imp_gap.dropna().sort_values("Change", ascending=False)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("2024 Import Potential", "2040 Needs — Import Potential"),
    horizontal_spacing=0.12,
    shared_yaxes=True,
)

top_imp = imp_gap.head(15)
fig.add_trace(
    go.Bar(
        y=top_imp["Country"],
        x=top_imp["Import Potential (%)_2024"],
        orientation="h",
        marker_color="#1f77b4",
        name="2024",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        y=top_imp["Country"],
        x=top_imp["Import Potential (%)_2040_Needs"],
        orientation="h",
        marker_color="#d62728",
        name="2040 Needs",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    height=500,
    title_text="Import Dependency Evolution: 2024 → 2040 Needs (Top 15)",
    template="plotly_white",
)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_xaxes(title_text="Import Potential (%)", row=1, col=1)
fig.update_xaxes(title_text="Import Potential (%)", row=1, col=2)
fig.show()

---
## 🗺️ Risk Analysis 6: Composite Risk Dashboard

In [ ]:
# ============================================================
# 19. COMPOSITE RISK DASHBOARD
# ============================================================
print("=" * 80)
print("🔌 EUROPEAN INTERCONNECTION RISK DASHBOARD")
print("=" * 80)

print("\n📡 INTERCONNECTOR INFRASTRUCTURE")
print(f"  Unique borders:           {interconnectors['Border'].nunique()}")
print(
    f"  Countries with links:     {pd.unique(interconnectors['From']).shape[0]} sending, {pd.unique(interconnectors['To']).shape[0]} receiving"
)
print(
    f"  Total Ref capacity:       {interconnectors[interconnectors['Scenario'] == 'Reference']['NTC_F'].sum():,.0f} MW"
)
print(
    f"  Total Needs capacity:     {interconnectors[interconnectors['Scenario'] == 'Needs']['NTC_F'].sum():,.0f} MW"
)
print(
    f"  Capacity gap:             {interconnectors[interconnectors['Scenario'] == 'Needs']['NTC_F'].sum() - interconnectors[interconnectors['Scenario'] == 'Reference']['NTC_F'].sum():,.0f} MW"
)

print("\n⚠️  IMPORT DEPENDENCY RISK")
crit_count = len(critical[critical["Import Potential (%)"] > 100]["Country"].unique())
print(f"  Critically dependent countries (>100%): {crit_count}")
print(
    f"  Average import potential (2024):        {import_potential[import_potential['Scenario'] == '2024']['Import Potential (%)'].mean():.1f}%"
)
print(
    f"  Average import potential (2040 Needs):  {import_potential[(import_potential['Target Year'] == 2040) & (import_potential['Scenario'] == 'Needs')]['Import Potential (%)'].mean():.1f}%"
)

print("\n🌱 RENEWABLE PENETRATION")
print(f"  European avg RES-E (2024): {res_by_country['RES-E Mean'].mean():.1f}%")
print(
    f"  Highest:                   {res_by_country.iloc[0]['Country']} ({res_by_country.iloc[0]['RES-E Mean']:.1f}%)"
)
print(
    f"  Lowest:                    {res_by_country.iloc[-1]['Country']} ({res_by_country.iloc[-1]['RES-E Mean']:.1f}%)"
)

print("\n🌊 CROSS-BORDER FLOWS")
print(f"  Unique flow corridors: {flow_summary.shape[0]}")
print(f"  Largest net exporter:  {net.iloc[0]['Country']} ({net.iloc[0]['Net_Flow']:.0f} MW)")
print(f"  Largest net importer:  {net.iloc[-1]['Country']} ({net.iloc[-1]['Net_Flow']:.0f} MW)")

print("\n📈 DEMAND GROWTH")
print(f"  Peak demand growth (avg 2024→2040): {peak_pivot['Growth_24_40'].mean():.1f}%")

print("\n" + "=" * 80)

---
## 🎯 Key Risk Takeaways

**For Infrastructure Risk Managers:**

1. **There is a significant capacity gap** — The difference between planned (Projects) interconnector capacity and required (Needs) capacity is substantial across major borders. FR-UK, AT-DE, and ES-FR corridors need the most expansion.

2. **Import dependency is critically high for several countries** — Montenegro (925%), Luxembourg (530%), and several Balkan countries face extreme import dependency. A disruption to key interconnectors would cause supply crises.

3. **Renewable leaders (DK, IE, LT, UK, PT) are also net importers** — High RES-E penetration creates intermittency risk: when wind/solar are low, these countries need significant import capacity to balance.

4. **The grid is asymmetric** — A small number of corridors (FR-UK, AT-DE, ES-FR, FI-SE, NO-SE) carry the vast majority of cross-border flows, creating single-point-of-failure risks.

5. **RES-E is projected to grow 15-35pp by 2040** — This will dramatically increase the need for interconnection to manage intermittency, yet current planning may not keep pace.

6. **Peak demand growth is uneven** — Some countries (likely those with high electrification) project 50%+ growth by 2040, while others see flat or declining demand.